# PARSHO: analyse one microscopy image without writing code

Use the **▶ play button** beside each numbered step, from top to bottom. After a step displays buttons or forms, make your selections before running the next step. **Do not use “Run all”**: image uploads and channel assignments need your input.

You can upload **one multichannel image** or **several aligned images containing separate channels** from the same field of view. No particular filenames are required. Tick which channels should be combined to find cells and which contain aggregates, autophagy puncta or RNA signals. A nucleus channel and a transfection marker are optional.

The notebook produces cell measurements, individual punctum measurements, inspection images, filtering decisions and shape-adapted radial distributions. It processes **one field of view**; optional positive/negative controls calibrate thresholds. Batch analysis and external-mask loading are covered by other tutorials.

**Before starting:** choose **Runtime → Change runtime type → T4 GPU** if available. CPU also works but cell segmentation can be slow. Installation and the first model download can take several minutes.

**Quick route:** install → upload and preview → assign channels/settings → skip controls unless needed → segment cells → analyse/export → download ZIP. All settings are form fields or checkboxes; leave advanced settings at their defaults initially.

## 1. Install and initialise

Run both cells below once per Colab session. Installation includes every optional image reader supported by PARSHO. If Colab requests a runtime restart, restart it, then run the import cell again.

In [ ]:
#@title 1a. Install PARSHO, all image readers and interactive controls
%pip install -q "parsho[microscopy-io] @ git+https://github.com/Fraternalilab/PARSHO.git" "cellpose>=4,<5" "ipywidgets>=8,<9" pandas

In [ ]:
#@title 1b. Initialise the analysis
from pathlib import Path
from importlib.metadata import version
import shutil
import tempfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Image as DisplayImage, clear_output
from google.colab import files, output
from cellpose import core, models

from parsho.colab_ui import AnalysisForm, ControlAssignment, FieldPicker
from parsho.colab_results import export_field_result, save_overlay
from parsho.single_image import (
    CELL_COLUMNS, OBJECT_COLUMNS, RADIAL_COLUMNS, FILTER_COLUMNS,
    DetectionSettings, analyze_field, combine_segmentation_channels, detect_signal,
)
from parsho.segmentation import find_optimal_threshold
from parsho.plotting import plot_radial_distribution
from parsho.provenance import runtime_provenance

output.enable_custom_widget_manager()
GPU = core.use_gpu()
WORK_DIR = Path(tempfile.mkdtemp(prefix="PARSHO_", dir=Path.cwd()))
model = None
form = None
segmentation_state = result_state = archive_path = None
positive_assignment = negative_assignment = None
print(f"Ready. GPU available: {GPU}")
print("Continue to step 2. The first segmentation run downloads the Cellpose model.")

## 2. Upload and preview your sample

Run the cell, click **Upload sample files**, and select either your single multichannel file or all the separate channel files together. They must describe the same field and have matching pixel dimensions and alignment.

Accepted formats: **TIFF/OME-TIFF, ND2, LIF, LOF, CZI, DICOM** (including extensionless DICOM), **PNG, JPEG and BMP**. Colour images expose their colour planes as channels. TIFF and vendor containers can contain multiple image series. JPEG intensities are affected by lossy compression; use the original microscopy files for quantitative work.

Click **Load and preview channels**. Most files need no further settings. Open the per-file advanced panel only to choose an image series, time point or Z plane. The default projects the Z stack at **one** time point. This is a 2-D analysis, not 3-D or time-series analysis.

For a TIFF or DICOM stack without clear dimension metadata, the loader will ask you to choose **Dimension order**: `CYX` = channels, `ZYX` = depth slices, `TYX` = time points; `Y` and `X` are height and width. Use the order reported by your acquisition/export software. All form indices start at **1**.

For large files already on Drive, tick the mount option and use the optional paths panel after granting Drive access.

For a first try, click **Try the supplied demo** instead of uploading. It loads the bundled cells/aggregates/nuclei images and prefills their roles in step 3; all settings remain editable.

In [ ]:
#@title 2. Select sample files, then click Load and preview channels
MOUNT_GOOGLE_DRIVE = False #@param {type:"boolean"}
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
sample_picker = FieldPicker("Sample")
segmentation_state = result_state = archive_path = None
display(sample_picker.widget)

## 3. Assign channel identities and choose analysis settings

Run the cell after previews appear. Work through the four tabs:

1. **Channel roles:** give measured channels meaningful names (e.g. “TRPV1”, “TRPA1”, “LC3”). Tick **Use for cell segmentation** for one or more channels and **Measure aggregates / puncta** for every signal to measure. The same channel can serve both purposes. Select a nucleus and/or transfection-marker channel from the dropdowns, or leave **None**.
2. **Cell segmentation:** choose how selected channels combine. Mean or maximum can combine any number; keeping channels separate supports up to three. Diameter `0` uses the model's automatic/default handling. [Cellpose settings reference](https://cellpose.readthedocs.io/en/latest/settings.html).
3. **Signal detection:** each measured signal has its own threshold method and minimum object area. Available methods are Otsu, percentile, local/adaptive, manual intensity or optional control calibration. Nuclear and transfection detection have their own settings. Unused parameters are disabled.
4. **Filtering, radial & export:** optionally exclude nuclear pixels, require signals or exclude border cells. Nuclear exclusion applies to both the puncta mask and analysed intensities. Radial analysis uses the nucleus centre when available, or the cell centre otherwise; you can explicitly choose either. A pixel width is optional and assumes square pixels.

**Without nuclei:** choose None. Nuclear exclusion/filtering is disabled, nuclear measurements remain blank, and automatic radial analysis uses the cell centre. **With nuclei:** exclusion is off by default; tick it only if nuclear fluorescence should not contribute to puncta measurements.

Cells without a detected nucleus can still have cell measurements. When using nucleus-centred radial analysis, those cells have no radial rows. The filtering table and `radial_included` column make this explicit.

All measurements retain the original signal intensities. Display/segmentation contrast adjustments do not change them. Defaults are starting points: always inspect the masks.

In [ ]:
#@title 3. Show channel roles and all analysis settings
sample_picker.require_loaded()
if form is None or form.source_channels is not sample_picker.channels:
    form = AnalysisForm(sample_picker)
    segmentation_state = result_state = archive_path = None
    positive_assignment = negative_assignment = None
    if sample_picker.is_demo:
        for row, name in zip(form.rows, ("Cells", "Aggregates", "Nuclei")):
            row["name"].value = name
        form.rows[0]["segmentation"].value = True
        form.rows[1]["signal"].value = True
        form.nucleus.value = 2
        print("Demo channel roles selected. Review the settings before continuing.")
else:
    print("Your settings are preserved for the currently loaded images.")
display(form.widget)

## 4. Optional positive/negative controls

**Skip this step unless a measured signal uses “Positive / negative controls” in step 3.** The two cells below safely print “Skipped” when controls are not selected.

Controls must be acquired with comparable staining, exposure/gain and intensity units. Each control can be one multichannel file or several separate channel images, using the same format support and preview controls as the sample.

Run **4a**, upload and preview both controls, then run **4b** to tick their segmentation channels and match their signal channels to the names used for the sample. A control nucleus is needed only when nuclear exclusion is enabled. Calibration uses the same minimum punctum area and nuclear-exclusion setting as the sample.

Control cell/nuclear previews are displayed and saved before threshold calibration. If calibration fails, inspect those previews and the printed paths before adjusting control assignments or thresholds.

In [ ]:
#@title 4a. Upload and preview controls, only when selected
_, measured_indices = form.selected()
control_names = [name for name, index in measured_indices.items()
                 if form.thresholds[index].fields["method"].value == "controls"]
positive_assignment = negative_assignment = None
control_request = (control_names, form.options["remove_nuclear"].value)
if control_names:
    positive_picker = FieldPicker("Positive control")
    negative_picker = FieldPicker("Negative control")
    display(positive_picker.widget, negative_picker.widget)
else:
    print("Skipped: no signal uses control calibration. Continue to step 5.")

In [ ]:
#@title 4b. Assign the control channels after loading both previews
if control_names:
    positive_assignment = ControlAssignment(
        positive_picker, control_names, use_nucleus=form.options["remove_nuclear"].value)
    negative_assignment = ControlAssignment(
        negative_picker, control_names, use_nucleus=form.options["remove_nuclear"].value)
    display(positive_assignment.widget, negative_assignment.widget)
else:
    print("Skipped: no control assignments are needed.")

## 5. Segment cells and check the outlines

Run this cell after completing the forms. Check whether the numbered outlines follow the cells. If necessary, adjust the **Cell segmentation** tab and rerun this step. Cell numbers are the original mask labels and remain the same in all tables and figures.

Changing input files requires rerunning steps 2 and 3. Changing segmentation channels or segmentation parameters requires rerunning step 5. Detection, filtering and radial settings can be adjusted by rerunning step 6.

In [ ]:
#@title 5. Run cell segmentation and display numbered outlines
segmentation_state = result_state = archive_path = None
seg_indices, measured_indices = form.selected()
configuration = form.snapshot()
seg_settings = configuration["segmentation"].copy()
combination = seg_settings.pop("combination")
diameter = seg_settings["diameter"]
if not np.isfinite([diameter, seg_settings["flow_threshold"], seg_settings["cellprob_threshold"]]).all() or diameter < 0 or seg_settings["flow_threshold"] < 0:
    raise ValueError("Diameter and flow threshold must be finite and nonnegative.")
seg_settings["diameter"] = None if diameter == 0 else diameter
seg_settings["normalize"] = False
segmentation_input = combine_segmentation_channels(
    [sample_picker.channels[index]["image"] for index in seg_indices], combination)
if model is None:
    print("Loading the default Cellpose-SAM model...")
    model = models.CellposeModel(gpu=GPU)
cell_masks, cell_flows, _ = model.eval(
    segmentation_input,
    channel_axis=-1 if segmentation_input.ndim == 3 else None,
    **seg_settings,
)
if not np.any(cell_masks):
    raise ValueError("No cells found. Check your channel choices; try lowering cell probability or minimum cell area, then rerun step 5.")
segmentation_state = dict(
    source_channels=sample_picker.channels, indices=seg_indices,
    parameters=configuration["segmentation"], masks=cell_masks,
    image=segmentation_input, model_parameters=seg_settings,
)
preview = WORK_DIR / "cell_segmentation.png"
display_image = segmentation_input.mean(axis=-1) if segmentation_input.ndim == 3 else segmentation_input
save_overlay(preview, display_image, cell_masks)
print(f"Detected {len(np.unique(cell_masks)) - int(np.any(cell_masks == 0))} cells. Inspect the outlines, then continue to step 6.")
display(DisplayImage(filename=str(preview)))

## 6. Analyse, inspect and prepare results

Run this cell to apply the current detection/filtering settings and calculate all measurements. Each ticked puncta channel is measured independently. The optional “keep cells with puncta” filter retains cells positive in **any** measured signal.

The output includes per-cell morphology, punctum counts and areas, individual punctum morphology and intensities, cell/nuclear signal totals, and radial coverage and intensity profiles. Shape-adapted bins follow the cell boundary; they are not circular rings. Radial intensity uses **detected puncta pixels**. Nuclear exclusion leaves the original raw-intensity columns available for comparison.

Check the displayed overlays (blue: nuclei; red: puncta; yellow: cell labels). To change thresholds or filters, edit step 3's form (rerunning the form cell also preserves choices for unchanged loaded images), then rerun this step. Each successful run creates a fresh results folder so old and new results cannot mix. With many cells, saving every radial figure can take time; its checkbox is in the export tab.

In [ ]:
#@title 6. Run analysis and save tables, masks and figures
result_state = archive_path = None
seg_indices, measured_indices = form.selected()
configuration = form.snapshot()
if (segmentation_state is None
        or segmentation_state["source_channels"] is not sample_picker.channels
        or segmentation_state["indices"] != seg_indices
        or segmentation_state["parameters"] != configuration["segmentation"]):
    raise ValueError("Inputs or segmentation settings changed. Rerun step 5 before analysis.")

channels = [item["image"] for item in sample_picker.channels]
signals = {name: channels[index] for name, index in measured_indices.items()}
nucleus_image = None if form.nucleus.value is None else channels[form.nucleus.value]
transfection_image = None if form.transfection.value is None else channels[form.transfection.value]
options = {key: widget.value for key, widget in form.options.items()}
if options["pixel_size_um"] == 0:
    options["pixel_size_um"] = None
if nucleus_image is None and options["radial"] and options["radial_center"] == "nucleus":
    raise ValueError("Choose Cell centre or Automatic, or assign a nucleus channel in step 3.")

needed_controls = [name for name, index in measured_indices.items()
                   if form.thresholds[index].fields["method"].value == "controls"]
calibrated, control_settings, control_previews = {}, {}, []
if needed_controls:
    if (positive_assignment is None or negative_assignment is None
            or control_request != (needed_controls, options["remove_nuclear"])):
        raise ValueError("Control selections changed or are missing. Complete steps 4a and 4b.")
    loaded_controls = []
    for title, assignment in [("positive", positive_assignment), ("negative", negative_assignment)]:
        mapping = assignment.snapshot()
        control_settings[title] = mapping
        control_channels = [item["image"] for item in assignment.picker.channels]
        control_input = combine_segmentation_channels(
            [control_channels[i] for i in mapping["segmentation"]],
            configuration["segmentation"]["combination"])
        control_masks, _, _ = model.eval(
            control_input, channel_axis=-1 if control_input.ndim == 3 else None,
            **segmentation_state["model_parameters"])
        if not np.any(control_masks):
            raise ValueError(f"No cells detected in the {title} control. Check its channel assignments.")
        nuclear_pixels = np.zeros(control_masks.shape, dtype=bool)
        if options["remove_nuclear"]:
            nuclear_pixels, _, _ = detect_signal(
                control_channels[mapping["nucleus"]], control_masks,
                form.nucleus_threshold.settings())
        loaded_controls.append((mapping, control_channels, control_masks, nuclear_pixels))
        control_previews.append((title, control_input, control_masks, nuclear_pixels))
        # Save and display before calibration, including if threshold search fails.
        control_preview = WORK_DIR / f"control_{title}.png"
        base = control_input.mean(axis=-1) if control_input.ndim == 3 else control_input
        save_overlay(control_preview, base, control_masks, nuclear_pixels)
        print(f"Inspect {title} control segmentation / nuclei: {control_preview}")
        display(DisplayImage(filename=str(control_preview)))
    for name in needed_controls:
        index = measured_indices[name]
        minimum = form.thresholds[index].fields["min_size"].value
        calibration_inputs = []
        for mapping, images, masks, nuclear_pixels in loaded_controls:
            image = images[mapping["signals"][name]]
            _, _, threshold = detect_signal(image, masks, DetectionSettings(min_size=minimum))
            calibration_inputs.append((image, masks, nuclear_pixels, float(threshold)))
        pos, neg = calibration_inputs
        try:
            calibrated[name], positive_pixels = find_optimal_threshold(
                pos[0], pos[1], neg[0], neg[1], pos[2], neg[2],
                t_min=min(pos[3], neg[3]), t_max=max(pos[3], neg[3]),
                min_size_px=minimum,
            )
        except ValueError as error:
            raise ValueError(f"Could not calibrate {name}: {error}. Inspect controls or choose another threshold method.") from error
        print(f"{name}: calibrated threshold {calibrated[name]:.4g}, {positive_pixels} positive-control pixels.")

detection = {name: form.thresholds[index].settings(calibrated.get(name))
             for name, index in measured_indices.items()}
result = analyze_field(
    segmentation_state["masks"], signals, detection=detection,
    nucleus=nucleus_image, nucleus_detection=form.nucleus_threshold.settings(),
    transfection=transfection_image, transfection_detection=form.transfection_threshold.settings(),
    **options,
)
configuration["controls"] = control_settings
configuration["calibrated_thresholds"] = {name: float(value) for name, value in calibrated.items()}
configuration["versions"] = {name: version(name) for name in ["parsho", "cellpose", "numpy", "scikit-image", "tifffile"]}
configuration["cellpose_model"] = str(model.pretrained_model)
configuration["provenance"] = runtime_provenance(model)
configuration["cellpose_gpu"] = bool(GPU)
run_root = Path(tempfile.mkdtemp(prefix="analysis_", dir=WORK_DIR))
results_dir = export_field_result(
    run_root / "PARSHO_results", result, signals, configuration,
    nucleus=nucleus_image, transfection=transfection_image,
    segmentation_image=segmentation_state["image"],
    save_all_radial=form.save_all_radial.value,
)
shutil.copy2(WORK_DIR / "cell_segmentation.png", results_dir / "cell_segmentation.png")
for title, control_input, control_masks, nuclear_pixels in control_previews:
    base = control_input.mean(axis=-1) if control_input.ndim == 3 else control_input
    save_overlay(results_dir / f"control_{title}.png", base, control_masks, nuclear_pixels)
print(f"Retained {len(result['retained_labels'])} of {len(result['filter_records'])} segmented cells.")
if not result["retained_labels"]:
    print("No cells passed the filters. Inspect cell_filtering.csv and relax the relevant filters in step 3.")
if options["radial"]:
    print(f"Radial centre: {result['center']}; cells with radial data per signal: "
          f"{ {name: len(items) for name, items in result['distributions'].items()} }")
for index, name in enumerate(signals, 1):
    print(f"Inspection: {name}")
    display(DisplayImage(filename=str(results_dir / f"signal_{index:02d}" / "retained_cells.png")))
for title, records, columns in [
    ("Cell measurements", result["cell_records"], CELL_COLUMNS),
    ("Individual puncta", result["object_records"], OBJECT_COLUMNS),
    ("Radial measurements", result["radial_records"], RADIAL_COLUMNS),
    ("Cell filtering", result["filter_records"], FILTER_COLUMNS),
]:
    print(f"{title}: {len(records)} rows. Preview shows the first 10; the download contains every row.")
    display(pd.DataFrame(records, columns=columns).head(10))
result_state = dict(result=result, configuration=form.snapshot(),
                    controls={title: assignment.snapshot() for title, assignment in
                              [("positive", positive_assignment), ("negative", negative_assignment)]}
                   if needed_controls else {}, directory=results_dir)
print("Analysis saved. Continue to step 7 to inspect radial plots, then step 8 to download everything.")

## 7. Inspect any cell's radial profile (optional)

Run this cell to display dropdowns for the signal and original cell label. The figure shows the analysed region, intensity image, aggregate coverage by radial bin and cumulative puncta intensity. If radial analysis was disabled or no cells qualify, it explains why instead of failing.

In [ ]:
#@title 7. Explore the radial profiles
if result_state is None:
    raise ValueError("Complete step 6 first.")
explore_result = result_state["result"]
signal_selector = widgets.Dropdown(options=list(explore_result["distributions"]), description="Signal:")
cell_selector = widgets.Dropdown(description="Cell label:")
radial_view = widgets.Output()
def show_radial(*_):
    with radial_view:
        clear_output(wait=True)
        name, cell_id = signal_selector.value, cell_selector.value
        if cell_id is None:
            print("No radial data for this signal. Check the radial checkbox, filters and centre selection in step 3.")
            return
        fig, _ = plot_radial_distribution(
            explore_result["cells"], explore_result["signal_masks"][name],
            explore_result["intensities"][name],
            explore_result["distributions"][name][cell_id],
            cell_name=f"{name} / cell {cell_id}", center_label=explore_result["center"].title(),
            show=False,
        )
        display(fig)
        plt.close(fig)
def choose_signal(*_):
    cell_selector.options = list(explore_result["distributions"][signal_selector.value])
    show_radial()
signal_selector.observe(choose_signal, names="value")
cell_selector.observe(show_radial, names="value")
display(widgets.HBox([signal_selector, cell_selector]), radial_view)
choose_signal()

## 8. Download all results — do this before closing Colab

Run the final cell. It creates a ZIP and starts a browser download. Allow downloads if your browser asks. If the download does not start, click **Download results ZIP** again, or open Colab's **Files** panel (folder icon on the left), find the exact path printed below, right-click the ZIP and choose **Download**.

The ZIP contains **all rows**, even though the notebook previews only ten. Colab session files are temporary; keep the ZIP on your computer or Drive.

| File | Contents |
| --- | --- |
| `summary.csv` | Cell counts, punctum counts, area and intensity totals for each signal |
| `cell_measurements.csv` | One row per retained cell per signal, including morphology, counts, areas, raw/analyzed intensities and radial inclusion |
| `aggregate_measurements.csv` | Individual puncta, with cell labels, size, shape, position and intensity |
| `cell_filtering.csv` | Every detected cell, which filters it passed and why it was excluded |
| `radial_distribution.csv` | Coverage, intensity, normalized shares and cumulative intensity for every cell and radial bin |
| `cell_masks.tif/.npy` | Labelled cells; labels match every figure and table |
| `signal_XX/` | Raw selected/projected signal, masks, analysed intensity, overlays and optional per-cell radial figures |
| `analysis_settings.json` | Input/channel mapping, all settings, thresholds and software versions |
| `READ_ME.txt` | Units, definitions, interpretation and how missing nuclear/radial data are represented |

Nucleus and transfection images/masks are included when supplied. Spatial measurements are in pixels unless you entered a pixel width; then extra area columns report square micrometres. Missing nuclear measurements are blank, not zero. There is no nucleus mask when no nuclear channel was supplied.

In [ ]:
#@title 8. Create the ZIP and download it
if result_state is None:
    raise ValueError("Complete step 6 before downloading.")
if result_state["configuration"] != form.snapshot():
    raise ValueError("Settings have changed since analysis. Rerun step 6 (and step 5 if segmentation changed), then download.")
if result_state["controls"]:
    current_controls = {title: assignment.snapshot() for title, assignment in
                        [("positive", positive_assignment), ("negative", negative_assignment)]}
    if current_controls != result_state["controls"]:
        raise ValueError("Control settings changed. Rerun step 6 before downloading.")
archive_path = Path(shutil.make_archive(
    str(result_state["directory"].parent / "PARSHO_results"), "zip",
    root_dir=result_state["directory"],
))
print(f"ZIP ready: {archive_path}")
print(f"Size: {archive_path.stat().st_size / 1024**2:.1f} MB")
print("Your download should start now. If it does not, click the button or use the Files panel → right-click ZIP → Download.")
download_button = widgets.Button(description="Download results ZIP", button_style="success",
                                 layout=widgets.Layout(width="260px"))
download_button.on_click(lambda _, path=archive_path: files.download(str(path)))
display(download_button)
files.download(str(archive_path))